In [1]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences




AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Load trained model
model = load_model("exercise_autoencoder.h5", compile=False)




In [3]:

# Mediapipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# Webcam
cap = cv2.VideoCapture(0)

# Buffer to hold recent frames (sliding window)
SEQUENCE_LENGTH = 30   # adjust (must be <= max_len used in training)
frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

# Threshold (you can tune this by testing)
THRESHOLD = 0.01 

In [4]:
def extract_keypoints(frame):
    """Extract 132 pose keypoints from a frame"""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    if results.pose_landmarks:
        landmarks = []
        for lm in results.pose_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z, lm.visibility])
        return np.array(landmarks, dtype=np.float32)
    return None

def check_exercise_quality(sequence):
    """Return feedback based on reconstruction error"""
    sequence_padded = pad_sequences([sequence], maxlen=100, dtype='float32', padding='post', truncating='post')
    
    pred = model.predict(sequence_padded, verbose=0)
    error = np.mean((sequence_padded - pred) ** 2)
    
    if error < THRESHOLD:
        return "Good ✅"
    else:
        return "Not Correct ❌"

In [ ]:
 



while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    keypoints = extract_keypoints(frame)
    if keypoints is not None:
        frame_buffer.append(keypoints)

    feedback = "Waiting..."  

    # Only run inference if we have enough frames
    if len(frame_buffer) == SEQUENCE_LENGTH:
        sequence = np.array(frame_buffer)
        feedback = check_exercise_quality(sequence)

    # Display feedback on frame
    cv2.putText(frame, feedback, (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0) if "Good" in feedback else (0, 0, 255), 2)

    cv2.imshow("Exercise Feedback", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import mediapipe as mp
import time

# === Load trained LSTM autoencoder ===
model = load_model("exercise_autoencoder.h5", compile=False)

# === Mediapipe Pose setup ===
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# === Webcam setup ===
cap = cv2.VideoCapture(0)
SEQUENCE_LENGTH = 30
frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

# === Thresholds (adjust after testing) ===
THRESHOLD_GOOD = 0.02
THRESHOLD_ALMOST = 0.05

def normalize_keypoints(landmarks):
    """Normalize pose keypoints to reduce variance due to position/scale."""
    landmarks = np.array(landmarks)
    landmarks = landmarks.reshape(-1, 4)
    # Normalize using the center of hips as reference
    if landmarks.shape[0] >= 24:
        hip_center = (landmarks[23][:3] + landmarks[24][:3]) / 2
        landmarks[:, :3] -= hip_center  # shift
    landmarks[:, :3] /= np.linalg.norm(landmarks[:, :3]) + 1e-8  # scale
    return landmarks.flatten()

def extract_keypoints(frame):
    """Extract pose keypoints (132 values) from frame."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    if results.pose_landmarks:
        landmarks = []
        for lm in results.pose_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z, lm.visibility])
        landmarks = normalize_keypoints(landmarks)
        return landmarks, results.pose_landmarks
    return None, None

def check_exercise_quality(sequence):
    """Calculate reconstruction error and return feedback & confidence."""
    sequence_padded = pad_sequences([sequence], maxlen=100, dtype='float32', padding='post', truncating='post')
    pred = model.predict(sequence_padded, verbose=0)
    error = np.mean((sequence_padded - pred) ** 2)

    # Convert error to confidence (inverted scale)
    confidence = max(0.0, 1.0 - (error / THRESHOLD_ALMOST))

    if error < THRESHOLD_GOOD:
        feedback = "Good ✅"
    elif error < THRESHOLD_ALMOST:
        feedback = "Almost 👍"
    else:
        feedback = "Not Correct ❌"

    return feedback, confidence, error

# === Main loop ===
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    keypoints, landmarks = extract_keypoints(frame)
    if keypoints is not None:
        frame_buffer.append(keypoints)

    feedback = "Waiting..."
    confidence = 0
    error = 0

    if len(frame_buffer) == SEQUENCE_LENGTH:
        sequence = np.array(frame_buffer)
        feedback, confidence, error = check_exercise_quality(sequence)

    # === Draw Pose Skeleton ===
    if landmarks:
        mp_drawing.draw_landmarks(frame, landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2))

    # === Display Feedback ===
    cv2.putText(frame, f"Feedback: {feedback}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 255, 0) if "Good" in feedback else (0, 165, 255) if "Almost" in feedback else (0, 0, 255), 2)
    cv2.putText(frame, f"Confidence: {confidence*100:.1f}%", (30, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    cv2.putText(frame, f"Reconstruction Error: {error:.4f}", (30, 130),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

    cv2.imshow("Exercise Quality Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

KeyboardInterrupt: 

: 